# Two-bunch console

The calls for a session at the machine. Cells only call `twobunch_deploy` functions; every call saves its record under `data/<date>_<stage>/`. Only the v2 sections write to the machine, through `actuate.py` after `arm()` (`tests/test_read_only.py`).

Tick each checklist in the cell, and copy the completed lists into the lab-notebook commissioning page with the session's records.

## Checklists

### Read-only release (per release)
- [ ] `pixi run test` passes locally and CI passes on the release commit
- [ ] `tests/test_read_only.py` passes: no write call in the package or the notebooks, PV names only in `pvs.py`
- [ ] the step readings equal the v7 replay of the DAQ runs, and the processing equals Yiheng's (the soft-IOC and DAQ tests ran, not skipped)
- [ ] every line of the release read
- [ ] the release's code hash (CI's `pixi run code-hash`) noted here: ...

### Before each session
- [ ] `check_env()`: code hash equals the release's; no import errors
- [ ] `check_pvs()`: all PVs answer
- [ ] `geometry_test()`: camera ROI and orientation as expected (`differs` understood)
- [ ] optics inside the policy's envelope (`advise.pre_shift()` below: FF quads ±15%, Q1FF ±5%, sextupoles ±35% of TEST_15668's), or a decision written down
- [ ] a background: taken with no beam, a pedestal, or a saved one
- [ ] `peek()`: the cut and centroids where expected
- [ ] TCAV voltage 18–32 MV and the readbacks separate off / + / − (`tcav_test()`)
- [ ] streak calibration from ops for this session, at its optics: value and sign, entered below; the cut polarity from the session's TCAV sign pairing ("+" as E300, "−" when reversed as on E331: check the jitter-slope sign)

### Measurement commissioning (once per setup)
- [ ] capture fraction and pulse-ID matching (`capture_test()`, `tcav_test()`)
- [ ] drop length after each flip
- [ ] whether the off reference can be reused between steps
- [ ] per-image and per-step noise against the env model
- [ ] drift per step against the trained drift
- [ ] readings inside the training range
- [ ] timing cycle chosen

## Session start

In [ ]:
from twobunch_deploy import session

session.check_env()

In [ ]:
session.check_pvs()

In [ ]:
session.geometry_test()

In [ ]:
session.optics_test()

In [ ]:
session.capture_test(seconds=10)

## Pre-shift (v2): the policy's envelope

`advise.pre_shift()` prints the ranges the policy was trained on beside the live values and what you enter, and saves them as the session's record (`data/<date>_V2/session_<HHMMSS>.json`). Nothing refuses: tick the list.

- [ ] ops: two-bunch beam with the witness trailing, 10 Hz if possible; `PREQ` feedback off; LI12–14 bunch-length trim off; energy feedbacks on (the LEMs stay on); plasma off
- [ ] every row ok, or a decision written down
- [ ] capture length set below from the beam rate: 15 s at 10 Hz, 30 s at 5 Hz (training: 15–35 kept frames per polarity, 20–45 s per step)
- [ ] free disk under `data/` for the shift: 5–10 MB a step
- [ ] the camera ROI contains the hand crop (`geometry_test()`)
- [ ] mover setpoints within ±0.75 mm (by hand: the runner doesn't read them)
- [ ] a start that fails the crop-edge check (edge fraction ≥ 0.01): lower the TCAV voltage, or the spacing, before arming

In [ ]:
from twobunch_deploy import advise

ENTERED = {"tcav_calibration_um_per_deg": None, "cut_polarity": "+", "charge_nC": None, "delay_ps": None,
           "pulse_lengths_ps": None, "budget": {"L1": 1.0, "L2": 1.5}, "step_scale": {"L1": 1.0, "L2": 1.0},
           "feedbacks": {"PREQ (BC11 bunch length)": "off", "LI12-14 bunch-length trim": "off", "energy": "on"}}
advise.pre_shift(ENTERED)

Optional: the continuous PV log (phases, bunch lengths, TCAV over the whole session, one file an hour in `data/<date>_V2/pvlog_*.npz`). Off unless started; stop it at the end.

In [ ]:
session.start_pv_log()

## Background

Run one: a background with no beam on DTOTR2, a pedestal with beam on, or a saved one (its `.npz`).

In [ ]:
background = session.take_background()
# background = session.pedestal()
# background = session.load_background("data/2026-11-04_S3/background_143016.npz")

## Looking at the frames

`peek()` is the newest frame; `peek_strip()` a few seconds of frames; `watch()` a `peek()` about once a second for 30 s (it holds the kernel: only between steps).

In [ ]:
session.peek(background)

In [ ]:
session.peek_strip(background, seconds=4)

In [ ]:
session.watch(background, seconds=30)

## The machine's PVs

In [ ]:
session.tcav_test(seconds=10)

In [ ]:
session.phases_test(seconds=60)

In [ ]:
session.drift_test(seconds=60)

## Steps

The streak calibration (µm on the screen per degree, at the live pixel size) comes from ops for this session: enter it, the capture length and the cut polarity before the first step.

In [ ]:
STREAK_UM_PER_DEG = None
SECONDS = 15            # 30 at 5 Hz
CUT_POLARITY = "+"      # "-" when the TCAV sign pairing is reversed (E331)

In [ ]:
r = session.step(background, streak_um_per_deg=STREAK_UM_PER_DEG, seconds=SECONDS, cut_polarity=CUT_POLARITY)
r.record["reading"], r.record["checks"]

In [ ]:
r.show()

In [ ]:
session.status()

## Writes (v2): armed sessions only

### Before arming
- [ ] ops agree to the session; who is at the console
- [ ] the `PREQ` feedback (BC11 bunch length) and the L2 bunch-length feedback off; the energy feedbacks on, so the LEMs stay on; all states recorded
- [ ] `pre_shift()` rows ok, or decisions written down
- [ ] streak calibration from ops
- [ ] the budget and step setting chosen inside the policy's envelope (budget L1 1–3°, L2 1.5–4°; step 0.3–1.5×, 0.5× or more under drift); start at ±1.0 / ±1.5 and 1×
- [ ] a step setting above 1× raises the per-write limit (at 1.5×: 0.375° L1, 0.45° L2): ops told
- [ ] detune moves count against the budget: keep them to half of it, as training assumed
- [ ] budget, step setting and limits read back from `arm()`
- [ ] arm at 1×; lower the step setting if the first proposals overshoot (the spacing passes the goal and comes back)

### End of session
- [ ] `undo_all()` or keep, stated in the e-log
- [ ] if keeping: ops reset each loop's setpoint to the current reading before re-enabling it
- [ ] feedbacks restored as agreed; `disarm()`; `session.stop_pv_log()` if started
- [ ] records copied back, the day's `_V2` and `_S5` folders and `data/goals.json`; then here: `python tools/session_table.py data/srv01/<date>_V2`

In [ ]:
from twobunch_deploy import actuate

actuate.arm("Ryan", budget={"L1": 1.0, "L2": 1.5}, step_scale={"L1": 1.0, "L2": 1.0})

In [ ]:
actuate.state()

In [ ]:
actuate.undo_all()

In [ ]:
actuate.disarm()

## Policy (v2): goals, shadow mode, approved cycles

The policy proposes L1 and L2 moves from each valid reading (`advise.py`). Use `advise.step()` here instead of `session.step()`: it remembers the readings for goals and proposals. Unarmed, `cycle()` is shadow mode: it proposes and logs, and writes nothing. Armed, it asks `apply? [y/N]` after each proposal: `y` applies it if answered within 60 s; anything else declines. An invalid reading gets no proposal. Manual moves are refused while an episode runs. Each episode's proposals and decisions go in `data/<date>_V2/episode_<HHMMSS_mmm>.jsonl`.

1. **Goal:** with the beam where ops want it, three valid steps, then `record_goal`. Goals stay in `data/goals.json` across days until replaced (recorded again under the same name) or removed (`remove_goal`); `use_goal` prints when each was recorded.
2. **Shadow:** `use_goal`, then cycles, unarmed.
3. **Approved:** `end_episode()`; arm (checklist above); detune with `move_l1` / `move_l2`, one step at a time inside the budget; `use_goal`; then cycles.
4. **Closed loop (only when Ryan decides):** `run()` applies each proposal without asking until a stop: any hold, the alert, a failed write, the budget used up on both controls, the end of its schedule, or Ctrl-C (a write in flight finishes first). It prints why it stopped; the episode stays, approved. A schedule `[(goal, n), ...]` moves to the next goal after n applied proposals, keeping the episode's history; after a stop, pass the rest it prints.
5. **Holds:** an acquisition fault (frames, reference, intensity): take another step. A beam-state fault (no cut, spacing under 60 µm or negative, crop edge) three times in a row raises the alert: wait, `end_episode()` and move by hand, or `undo_all()`.
6. **End:** `end_episode()`, then the end-of-session checklist above.

In [ ]:
from twobunch_deploy import advise

for _ in range(3):
    r = advise.step(background, streak_um_per_deg=STREAK_UM_PER_DEG, seconds=SECONDS, cut_polarity=CUT_POLARITY)
    print(r.record["valid"], r.record["reading"])

In [ ]:
advise.record_goal("A")

In [ ]:
advise.remove_goal("A")

In [ ]:
actuate.move_l1(0.25)

In [ ]:
actuate.move_l2(0.3)

In [ ]:
advise.use_goal("A")

In [ ]:
for _ in range(10):
    advise.cycle(background, streak_um_per_deg=STREAK_UM_PER_DEG, seconds=SECONDS, cut_polarity=CUT_POLARITY)

In [ ]:
advise.run(background, STREAK_UM_PER_DEG, seconds=SECONDS, cut_polarity=CUT_POLARITY)

In [ ]:
advise.run(background, STREAK_UM_PER_DEG, schedule=[("A", 16), ("B", 16)], seconds=SECONDS, cut_polarity=CUT_POLARITY)

In [ ]:
advise.state()

In [ ]:
advise.end_episode()

In [ ]:
session.stop_pv_log()